# 03. Data Cleaning (QA Dataset + Audit Dataset)

Cleans and merges the raw extracted data into the two analysis-ready datasets:
- `data/cleaned/qa_defect_dataset.csv`
- `data/cleaned/audit_disclosure_dataset.csv`

**Fixes applied vs. the original scaffold version of this script:**
1. `clean_audit_dataset()` now renames PCAOB's real column names (e.g. `"Global
   Network"`, `"Auditing Standard"`) to the names `05_rq3_pcaob_severity_model.ipynb`
   actually expects. The original left them unrenamed, which would have caused a
   `KeyError` downstream.
2. `clean_audit_dataset()` now **deliberately excludes** `"Audit Area"` and several
   other PCAOB fields that are structurally populated **only** for Part I.A
   records: 100% missing for every real Part I.B record in this dataset.
   Including them lets a model trivially "predict" severity from a
   data-collection artifact rather than real engagement characteristics. This
   was caught as a genuine data-leakage bug during RQ3 modeling (it produced a
   suspicious 100% accuracy) and is fixed here at the cleaning stage, so it can
   never re-enter downstream modeling.


In [1]:
!pip install -q pandas numpy || pip install -q pandas numpy --break-system-packages

In [2]:
import pandas as pd
import numpy as np
import os

for _d in ["../data/raw", "../data/cleaned"]:
    os.makedirs(_d, exist_ok=True)

ERA_CUTOFF = "2023-01-01"  # boundary between pre-AI-coding and AI-assisted-coding eras

# Fields verified to be structurally leaked (100% missing for Part I.B records,
# 0% missing for Part I.A) -- NEVER use these as model features.
LEAKED_PCAOB_FIELDS = [
    "Issuer Reference Key",
    "Firm Played A Role But Was Not The Lead Auditor",
    "Audits Affected By The Deficiencies Identified In Part IA",
    "Classification Of Audits With Part IA Deficiencies",
    "Audit Area",
    "Firm-Identified Risk Assessment",
    "Paragraph Of The Auditing Standard",
    "Description In The Firm's Inspection Report",
]

GITHUB_BASE = "https://raw.githubusercontent.com/daljeetkaurJohar/qm640-governance-analytics/master"
import urllib.request

def ensure_file(local_path, github_relative_path):
    if os.path.exists(local_path):
        print(f"Found locally: {local_path}")
        return True
    os.makedirs(os.path.dirname(local_path) or ".", exist_ok=True)
    url = f"{GITHUB_BASE}/{github_relative_path}"
    try:
        print(f"Not found locally -- fetching from GitHub repo: {url}")
        req = urllib.request.Request(url, headers={"User-Agent": "qm640-capstone"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            content = resp.read()
        with open(local_path, "wb") as f:
            f.write(content)
        print(f"Downloaded {len(content)} bytes from GitHub -> {local_path}")
        return True
    except Exception as e:
        print(f"GitHub fetch failed ({e}).")
        return False


## Clean the QA / defect dataset (Apache JIRA)

In [3]:
def clean_qa_dataset():
    ensure_file("../data/raw/apache_jira_raw.csv", "data/raw/apache_jira_raw.csv")
    df = pd.read_csv("../data/raw/apache_jira_raw.csv")
    df = df.drop_duplicates(subset="issue_id")

    df["resolution_date"] = pd.to_datetime(df["resolution_date"], errors="coerce")
    df = df.dropna(subset=["resolution_date"])

    df["era"] = np.where(df["resolution_date"] < ERA_CUTOFF, "pre_ai", "ai_era")

    df["created"] = pd.to_datetime(df["created"], errors="coerce")
    df["resolution_time_days"] = (df["resolution_date"] - df["created"]).dt.days
    df = df[df["resolution_time_days"] >= 0]

    # NOTE: loc, cyclomatic_complexity, prior_defect_count, test_coverage_pct, and
    # defect_prone still need real repository-mining (e.g. `lizard`/`radon` against
    # git history) -- genuinely not yet done, see interim report Limitations.
    for col in ["loc", "cyclomatic_complexity", "prior_defect_count",
                "test_coverage_pct", "defect_prone"]:
        if col not in df.columns:
            df[col] = np.nan

    df.to_csv("../data/cleaned/qa_defect_dataset.csv", index=False)
    n_pre = (df["era"] == "pre_ai").sum()
    n_ai = (df["era"] == "ai_era").sum()
    print(f"Cleaned QA/defect dataset: {len(df)} real records ({n_pre} pre-AI-era, {n_ai} AI-era)")
    return df

qa_df = clean_qa_dataset()

Not found locally -- fetching from GitHub repo: https://raw.githubusercontent.com/daljeetkaurJohar/qm640-governance-analytics/master/data/raw/apache_jira_raw.csv
Downloaded 2894682 bytes from GitHub -> ../data/raw/apache_jira_raw.csv
Cleaned QA/defect dataset: 30733 real records (25327 pre-AI-era, 5406 AI-era)


## Clean the audit / disclosure dataset (PCAOB) -- leakage-fixed

In [4]:
def clean_audit_dataset():
    ensure_file("../data/raw/pcaob_deficiencies_raw.csv", "data/raw/pcaob_deficiencies_raw.csv")
    pcaob = pd.read_csv("../data/raw/pcaob_deficiencies_raw.csv")
    pcaob = pcaob.drop_duplicates(subset="deficiency_id")

    pcaob["severity_part"] = pcaob["severity_part"].apply(
        lambda x: 1 if str(x).strip().upper() in ("I.A", "1.A", "IA") else 0
    )

    pcaob = pcaob.rename(columns={
        "Global Network": "firm_network_category",
        "Auditing Standard": "standard_cited",
        "Inspection Year": "inspection_year",
        "Inspection Type": "inspection_type",
        "Country": "country",
        "Finding Count": "finding_count",
    })
    pcaob["firm_network_category"] = pcaob["firm_network_category"].fillna(
        "Independent/Unaffiliated"
    )

    # DROP the leaked, Part-I.A-only fields entirely.
    pcaob = pcaob.drop(columns=[c for c in LEAKED_PCAOB_FIELDS if c in pcaob.columns])

    pcaob.to_csv("../data/cleaned/audit_disclosure_dataset.csv", index=False)
    print(f"Cleaned audit/disclosure dataset: {len(pcaob)} real PCAOB records")
    print(f"Leaked fields removed: {LEAKED_PCAOB_FIELDS}")
    return pcaob

audit_df = clean_audit_dataset()

GITHUB_BASE = "https://raw.githubusercontent.com/daljeetkaurJohar/qm640-governance-analytics/master"
import urllib.request

def ensure_file(local_path, github_relative_path):
    if os.path.exists(local_path):
        print(f"Found locally: {local_path}")
        return True
    os.makedirs(os.path.dirname(local_path) or ".", exist_ok=True)
    url = f"{GITHUB_BASE}/{github_relative_path}"
    try:
        print(f"Not found locally -- fetching from GitHub repo: {url}")
        req = urllib.request.Request(url, headers={"User-Agent": "qm640-capstone"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            content = resp.read()
        with open(local_path, "wb") as f:
            f.write(content)
        print(f"Downloaded {len(content)} bytes from GitHub -> {local_path}")
        return True
    except Exception as e:
        print(f"GitHub fetch failed ({e}).")
        return False


Not found locally -- fetching from GitHub repo: https://raw.githubusercontent.com/daljeetkaurJohar/qm640-governance-analytics/master/data/raw/pcaob_deficiencies_raw.csv
Downloaded 11214050 bytes from GitHub -> ../data/raw/pcaob_deficiencies_raw.csv
Cleaned audit/disclosure dataset: 16704 real PCAOB records
Leaked fields removed: ['Issuer Reference Key', 'Firm Played A Role But Was Not The Lead Auditor', 'Audits Affected By The Deficiencies Identified In Part IA', 'Classification Of Audits With Part IA Deficiencies', 'Audit Area', 'Firm-Identified Risk Assessment', 'Paragraph Of The Auditing Standard', "Description In The Firm's Inspection Report"]
